# ⚙️ Notebook 05: Preprocessing
## Customer Churn Prediction
- One-Hot Encoding
- Feature Scaling
- Train/Test Split
- SMOTE for Imbalance
- Save Preprocessor

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
print("✅ Libraries imported!")

if os.path.exists('/kaggle/working'):
    DATA_PATH = '/kaggle/working/data/processed/featured_data.csv'
    MODEL_DIR = Path('/kaggle/working/models')
else:
    DATA_PATH = Path('../data/processed/featured_data.csv')
    MODEL_DIR = Path('../models')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded: {df.shape}")

In [ ]:
X = df.drop(['customerid', 'churn'], axis=1)
y = df['churn'].map({'Yes': 1, 'No': 0})
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns
print(f"🔢 Numeric: {list(numeric_features)}")
print(f"🔠 Categorical: {list(categorical_features)}")

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)])
print("✅ Preprocessor defined")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✅ Train Size: {X_train.shape}, Test Size: {X_test.shape}")

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f"🔄 After SMOTE - Train Size: {X_train_resampled.shape}")
print(f"📊 Class Dist (Resampled): {pd.Series(y_train_resampled).value_counts().to_dict()}")

In [ ]:
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train_resampled)
X_test_processed = preprocessor.transform(X_test)
print(f"✅ Processed Shapes: Train {X_train_processed.shape}, Test {X_test_processed.shape}")

In [ ]:
joblib.dump(preprocessor, MODEL_DIR / 'preprocessor.pkl')
print(f"💾 Saved preprocessor to: {MODEL_DIR / 'preprocessor.pkl'}")
if os.path.exists('/kaggle/working'):
    np.save('/kaggle/working/X_train.npy', X_train_processed)
    np.save('/kaggle/working/X_test.npy', X_test_processed)
    np.save('/kaggle/working/y_train.npy', y_train_resampled)
    np.save('/kaggle/working/y_test.npy', y_test)
    print("💾 Saved numpy arrays for next notebooks")
print("✅ Notebook 05 Complete!")